# Pipeline completo a MongoDB Atlas

Proyecto 2 - Análisis Semántico de Reseñas

Este notebook hace todo de una vez, contra Atlas:

1. Migra el corpus (`corpus_limpio.csv`) con POS tags, métricas y polaridad
2. Sube los embeddings de Word2Vec y BETO
3. Corre la comparación TF-IDF vs Word2Vec vs BETO (clasificación y clustering)
4. Guarda el resumen de métricas también en Atlas

Requisitos en la misma carpeta: `corpus_limpio.csv`, `skipgram_model.bin`, `beto_embeddings.npy`, y un archivo `.env` con `MONGO_URI`.

## 0. Conexión a Atlas

In [1]:
import os
from datetime import datetime

import numpy as np
import pandas as pd
import spacy
from dotenv import load_dotenv
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from pymongo import MongoClient, UpdateOne
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

load_dotenv()
client = MongoClient(os.environ["MONGO_URI"])
db = client["resenas_costa_rica"]
coleccion = db["resenas"]

print("Conectado a:", db.name)


Conectado a: resenas_costa_rica


## 1. Migrar el corpus

Mismo proceso que `migrar_corpus.py`: limpia el texto, saca POS tags con spaCy y NLTK,
calcula métricas y polaridad, y sube todo a la colección `resenas`.

In [2]:
CONTENT_POS = {"NOUN", "VERB", "ADJ", "ADV"}


def limpiar_texto(texto):
    if not isinstance(texto, str):
        return ""
    t = texto.lower()
    import re
    t = re.sub(r"[^a-záéíóúüñ\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def calcular_polaridad(calificacion):
    if calificacion is None:
        return None
    if calificacion <= 2:
        return "negativa"
    return "neutral" if calificacion == 3 else "positiva"


def calcular_metricas(doc_spacy):
    tokens = [t for t in doc_spacy if not t.is_punct and not t.is_space]
    n = len(tokens)
    if n == 0:
        return {"num_palabras": 0, "densidad_lexica": None,
                "ratio_sustantivos_verbos": None, "densidad_adjetivos": None}

    conteo = {}
    for t in tokens:
        conteo[t.pos_] = conteo.get(t.pos_, 0) + 1

    num_contenido = sum(conteo.get(p, 0) for p in CONTENT_POS)
    num_verb = conteo.get("VERB", 0)

    return {
        "num_palabras": n,
        "densidad_lexica": round(num_contenido / n, 4),
        "ratio_sustantivos_verbos": round(conteo.get("NOUN", 0) / num_verb, 4) if num_verb else None,
        "densidad_adjetivos": round(conteo.get("ADJ", 0) / n, 4),
    }


In [3]:
df = pd.read_csv("corpus_limpio.csv")
df["fecha"] = pd.to_datetime(df["fecha"], format="%m/%d/%Y", errors="coerce")
df["fecha"] = df["fecha"].astype(object).where(df["fecha"].notna(), None)
df["calificacion"] = pd.to_numeric(df["calificacion"], errors="coerce")

print(f"Filas en el corpus: {len(df)}")

nlp = spacy.load("es_core_news_md")
textos_limpios = [limpiar_texto(t) for t in df["texto"]]
docs_spacy = list(nlp.pipe(textos_limpios, batch_size=50))


Filas en el corpus: 2116


In [4]:
documentos = []

for (_, fila), texto_limpio, doc_spacy in zip(df.iterrows(), textos_limpios, docs_spacy):
    tokens_nltk = word_tokenize(texto_limpio, preserve_line=True) if texto_limpio else []
    pos_nltk = [[w, tag] for w, tag in pos_tag(tokens_nltk)] if tokens_nltk else []
    pos_spacy = [[t.text, t.pos_, t.tag_, t.lemma_] for t in doc_spacy]
    calificacion = float(fila["calificacion"]) if pd.notna(fila["calificacion"]) else None

    documentos.append({
        "texto": fila["texto"],
        "texto_limpio": texto_limpio,
        "calificacion": calificacion,
        "tipo_lugar": fila.get("tipo_lugar"),
        "lugar": fila.get("lugar"),
        "fuente": fila.get("fuente"),
        "fecha": fila["fecha"],
        "fecha_recopilacion": datetime.utcnow(),
        "polaridad": calcular_polaridad(calificacion),
        "idioma": None,
        "url_fuente": None,
        "pos_tags": {"nltk": pos_nltk, "spacy": pos_spacy},
        "embeddings": {"word2vec_avg": [], "beto_cls": []},
        "metricas": calcular_metricas(doc_spacy),
    })

print(f"Documentos armados: {len(documentos)}")


C:\Users\steven16\AppData\Local\Temp\ipykernel_42680\1882552133.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fecha_recopilacion": datetime.utcnow(),


Documentos armados: 2116


In [5]:
borrados = coleccion.delete_many({})
print(f"Documentos borrados en Atlas: {borrados.deleted_count}")

resultado = coleccion.insert_many(documentos)
print(f"Documentos insertados en Atlas: {len(resultado.inserted_ids)}")


Documentos borrados en Atlas: 1
Documentos insertados en Atlas: 2116


## 2. Subir embeddings

Word2Vec: se calcula el promedio de los vectores de cada reseña con el modelo ya entrenado.
BETO: se reutiliza `beto_embeddings.npy` (ya calculado en el notebook 04), en el mismo
orden que este mismo CSV, así que se empareja directo por posición.

In [6]:
wv = Word2Vec.load("skipgram_model.bin").wv
beto_embeddings = np.load("beto_embeddings.npy")

assert len(beto_embeddings) == len(df), "El .npy no tiene el mismo tamaño que el CSV"


def vector_promedio(tokens, wv):
    vectores = [wv[t] for t in tokens if t in wv]
    if not vectores:
        return np.zeros(wv.vector_size)
    return np.mean(vectores, axis=0)


tokens_por_fila = [t.split() for t in textos_limpios]
w2v_embeddings = np.array([vector_promedio(tok, wv) for tok in tokens_por_fila])

print("Word2Vec:", w2v_embeddings.shape)
print("BETO:", beto_embeddings.shape)


Word2Vec: (2116, 100)
BETO: (2116, 768)


In [7]:
operaciones = []
for doc_id, v_w2v, v_beto in zip(resultado.inserted_ids, w2v_embeddings, beto_embeddings):
    operaciones.append(UpdateOne(
        {"_id": doc_id},
        {"$set": {
            "embeddings.word2vec_avg": v_w2v.tolist(),
            "embeddings.beto_cls": v_beto.tolist(),
        }}
    ))

for i in range(0, len(operaciones), 500):
    coleccion.bulk_write(operaciones[i:i + 500])

print(f"Embeddings subidos: {len(operaciones)}")


Embeddings subidos: 2116


## 3. Comparación TF-IDF vs Word2Vec vs BETO

Clasificación de polaridad (positiva 4-5⭐ vs negativa 1-3⭐) y clustering,
con el mismo split de train/test para los tres métodos.

In [8]:
tfidf_vec = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
bow_embeddings = tfidf_vec.fit_transform(df["texto"]).toarray()

print(f"TF-IDF: {bow_embeddings.shape}")
print(f"Vocabulario: {len(tfidf_vec.vocabulary_)}")


TF-IDF: (2116, 1000)
Vocabulario: 1000


In [9]:
y = (df["calificacion"] >= 4).astype(int)
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

representaciones = {
    "TF-IDF": bow_embeddings,
    "Word2Vec": w2v_embeddings,
    "BETO": beto_embeddings,
}

resultados_clasificacion = []

for nombre, X in representaciones.items():
    X_train, X_test = X[train_idx], X[test_idx]
    modelo = LogisticRegression(random_state=42, max_iter=1000)
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    resultados_clasificacion.append({
        "Método": nombre,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
    })

df_clasificacion = pd.DataFrame(resultados_clasificacion)
df_clasificacion


,Método,Accuracy,Precision,Recall,F1
0,TF-IDF,0.893868,0.892086,1.000000,0.942966
1,Word2Vec,0.903302,0.916877,0.978495,0.946684
2,BETO,0.896226,0.931579,0.951613,0.941489


In [10]:
resultados_clustering = []

for nombre, X in representaciones.items():
    kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    score = silhouette_score(X, labels)
    resultados_clustering.append({
        "Método": nombre,
        "Silhouette Score": score,
        "Interpretación": "Débil" if score < 0.3 else "Aceptable" if score < 0.5 else "Bueno",
    })

df_clustering = pd.DataFrame(resultados_clustering)
df_clustering


,Método,Silhouette Score,Interpretación
0,TF-IDF,0.008628,Débil
1,Word2Vec,0.062355,Débil
2,BETO,0.058579,Débil


In [11]:
df_caracteristicas = pd.DataFrame({
    "Método": ["TF-IDF", "Word2Vec", "BETO"],
    "Dimensiones": [bow_embeddings.shape[1], w2v_embeddings.shape[1], beto_embeddings.shape[1]],
    "Semántica": ["No", "Estática", "Contextual"],
    "Polisemia": ["No", "No (1 vector por palabra)", "Sí"],
})
df_caracteristicas


,Método,Dimensiones,Semántica,Polisemia
0,TF-IDF,1000,No,No
1,Word2Vec,100,Estática,No (1 vector por palabra)
2,BETO,768,Contextual,Sí


## 4. Guardar el resumen en Atlas

In [12]:
resumen = {
    "fecha_generado": datetime.utcnow(),
    "total_resenas": len(df),
    "clasificacion": df_clasificacion.to_dict(orient="records"),
    "clustering": df_clustering.to_dict(orient="records"),
    "caracteristicas": df_caracteristicas.to_dict(orient="records"),
    "mejor_clasificacion": df_clasificacion.loc[df_clasificacion["Accuracy"].idxmax(), "Método"],
    "mejor_clustering": df_clustering.loc[df_clustering["Silhouette Score"].idxmax(), "Método"],
}

db["comparacion_metodos"].delete_many({})
db["comparacion_metodos"].insert_one(resumen)

print("Guardado en la colección 'comparacion_metodos'")
print(f"Mejor en clasificación: {resumen['mejor_clasificacion']}")
print(f"Mejor en clustering: {resumen['mejor_clustering']}")


C:\Users\steven16\AppData\Local\Temp\ipykernel_42680\2162465033.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fecha_generado": datetime.utcnow(),


Guardado en la colección 'comparacion_metodos'
Mejor en clasificación: Word2Vec
Mejor en clustering: Word2Vec


## 5. Verificación final

In [13]:
print("resenas:", coleccion.count_documents({}))
print("con word2vec:", coleccion.count_documents({"embeddings.word2vec_avg": {"$ne": []}}))
print("con beto:", coleccion.count_documents({"embeddings.beto_cls": {"$ne": []}}))
print("comparacion_metodos:", db["comparacion_metodos"].count_documents({}))


resenas: 2116
con word2vec: 2116
con beto: 2116
comparacion_metodos: 1
